In [1]:
import os
import time
import glob
import pandas as pd
from Bio import Entrez
import xml.etree.ElementTree as ET

In [2]:
Entrez.email = "example@gmail.com"

In [3]:
PILLAR_KEYWORDS = {
    "01": '"unfolded protein response" OR "ER stress" OR "mitochondrial stress" OR "lysosome stress"',
    "02": '"heat shock" OR "chaperone" OR "autophagosome" OR "autophagy"',
    "03": '"DNA damage" OR "DNA repair" OR "p53" OR "senescence"',
    "04": '"hypoxia" OR "amino acid starvation" OR "nutrient deprivation" OR "AMPK" OR "mTOR"',
    "05": '"ubiquitin" OR "E3 ligase" OR "proteasome"',
    "06": '"mechanotransduction" OR "PIEZO" OR "focal adhesion" OR "integrin"',
    "07": '"mitochondria-associated membrane" OR "MAM" OR "organelle contact site" OR "membrane tether"',
    "08": '"ribosome collision" OR "ribotoxic stress" OR "p38" OR "JNK"',
    "09": '"cGAS" OR "STING" OR "inflammasome" OR "nucleic acid sensing"',
    "10": '"caspase" OR "ferroptosis" OR "gasdermin" OR "cuproptosis" OR "cell death execution"'
}

In [4]:
def fetch_tier1_pmid_and_count(gene_name, stress_terms):
    if pd.isna(gene_name) or str(gene_name).strip() == "":
        return "", 0
    
    primary_gene = str(gene_name).split()[0]
    
    # Species & Disease Matrix Filters
    species_filter = '(humans[MH] OR mice[MH] OR mouse[TIAB] OR human[TIAB] OR "murine"[TIAB])'
    disease_filter = (
        '('
        '"neoplasms"[MH] OR cancer* OR tumor* OR malignan* OR carcinoma OR leukemia OR lymphoma OR oncology OR '
        '"neurodegenerative diseases"[MH] OR neurodegenerat* OR alzheimer* OR parkinson* OR dementia OR "amyotrophic lateral sclerosis" OR '
        '"growth and development"[SH] OR developmen* OR "embryonic development" OR "cell differentiation" OR '
        '"cardiovascular diseases"[MH] OR heart* OR cardiac OR cardiotox* OR myocardial OR cardiomyopathy OR '
        '"diabetes mellitus"[MH] OR diabetes OR diabetic OR insulin-resistan*'
        ')'
    )
    
    query = f"({primary_gene}[TIAB]) AND ({stress_terms}[TIAB]) AND {species_filter} AND {disease_filter}"
    
    try:
        # Scan top 5 hits locally, but check total counts
        handle = Entrez.esearch(db="pubmed", term=query, retmax=5)
        record = Entrez.read(handle)
        handle.close()
        
        # Extract the total number of matching papers from NCBI natively
        total_paper_count = int(record.get("Count", 0))
        
        id_list = record["IdList"]
        if not id_list:
            return "", 0
            
        for pmid in id_list:
            try:
                # Local verification density filters
                summary_handle = Entrez.esummary(db="pubmed", id=pmid)
                summary_record = Entrez.read(summary_handle)
                summary_handle.close()
                
                fetch_handle = Entrez.efetch(db="pubmed", id=pmid, retmode="xml")
                xml_data = fetch_handle.read()
                fetch_handle.close()
                
                root = ET.fromstring(xml_data)
                
                title_node = root.find(".//ArticleTitle")
                title_text = title_node.text if title_node is not None and title_node.text else ""
                
                abstract_nodes = [node.text for node in root.findall(".//AbstractText") if node.text]
                abstract_text = " ".join(abstract_nodes)
                                
                # Local Density Check
                gene_count = abstract_text.upper().count(primary_gene.upper())
                if gene_count < 3:
                    continue 
                
                clean_stress_query = stress_terms.replace('"', '').upper()
                keywords_to_check = [k.strip() for k in clean_stress_query.split("OR")]
                
                keyword_in_title = any(k in title_text.upper() for k in keywords_to_check)
                total_keyword_abstract_count = sum(abstract_text.upper().count(k) for k in keywords_to_check)
                
                if keyword_in_title or total_keyword_abstract_count >= 2:
                    # Return BOTH the verified PMID and the total macro paper count
                    return pmid, total_paper_count
                
            except Exception:
                continue
                
        # If no single paper passes local density, it doesn't get a Tier 1 PMID
        return "", 0 
        
    except Exception:
        return "", 0

In [5]:
data_folder = "../data/"
tsv_files = glob.glob(os.path.join(data_folder, "*.tsv"))

In [6]:
for file_path in tsv_files:
    file_name = os.path.basename(file_path)
    if file_name.startswith("Validated_"): continue

    matching_pillar = next((p for p in PILLAR_KEYWORDS if p in file_name), None)
    if not matching_pillar: continue
        
    stress_query = PILLAR_KEYWORDS[matching_pillar]
    print(f"\nProcessing {file_name}...")

    # Read the TSV raw with no assumptions about headers
    df_raw = pd.read_csv(file_path, sep='\t', header=None)
    
    # Find the row index that contains the real table columns
    header_idx = None
    for idx, row in df_raw.iterrows():
        row_values = [str(val) for val in row.values]
        if any("Gene Names" in val or "Protein names" in val for val in row_values):
            header_idx = idx
            break
            
    if header_idx is None:
        print(f"Could not find data headers in {file_name}. Skipping.")
        continue

    # Re-read the file properly, cutting out everything above the real header
    df = pd.read_csv(file_path, sep='\t', skiprows=header_idx)
    df.columns = [str(col).strip() for col in df.columns]
    
    # Find target Gene column dynamically
    gene_col = next((col for col in df.columns if "Gene" in col and "Name" in col), None)
    
    if not gene_col:
        print(f"Missing Gene Names column. Available columns: {df.columns.tolist()}")
        continue
        
    print(f"Found headers at row {header_idx}. Target column: '{gene_col}'")
    
    # Track both IDs and Total Frequency Counts
    pmids = []
    paper_counts = []
    
    # Loop through every gene row
    for index, row in df.iterrows():
        gene = row[gene_col]
        
        # Skip if the row is blank
        if pd.isna(gene) or str(gene).strip() == "" or str(gene).lower() == 'nan':
            pmids.append("")
            paper_counts.append(0)
            continue
            
        # Clean the string to grab the official symbol
        primary_gene = str(gene).split()[0]
            
        print(f"Checking {primary_gene}...", end="\r")
        
        # Call the updated function that yields a tuple (PMID, total_count)
        found_pmid, found_count = fetch_tier1_pmid_and_count(primary_gene, stress_query)
        
        pmids.append(found_pmid)
        paper_counts.append(found_count)
        time.sleep(0.35)
        
    # columns mapped to the individual output file 
    df["Supporting_PMID"] = pmids
    df["Paper_Count"] = paper_counts
    
    # Sort files locally by giving priority to valid hits
    df["has_pmid"] = df["Supporting_PMID"].astype(str).str.strip() != ""
    df = df.sort_values(by="has_pmid", ascending=False).drop(columns=["has_pmid"])
    
    # Write back cleanly
    output_path = os.path.join("../output/", f"Validated_{file_name}")
    df.to_csv(output_path, sep='\t', index=False)
    print(f"Saved and sorted: {output_path}")


Processing 10_MoleculeRelatedStress.tsv...
Found headers at row 3. Target column: 'Gene Names'
Saved and sorted: ../output/Validated_10_MoleculeRelatedStress.tsv

Processing 03_GenetoxicStress.tsv...
Found headers at row 3. Target column: 'Gene Names'
Saved and sorted: ../output/Validated_03_GenetoxicStress.tsv

Processing 01_OrganelleStress.tsv...
Found headers at row 3. Target column: 'Gene Names'
Saved and sorted: ../output/Validated_01_OrganelleStress.tsv

Processing 05_ProteasomalUbiquitin.tsv...
Found headers at row 3. Target column: 'Gene Names'
Saved and sorted: ../output/Validated_05_ProteasomalUbiquitin.tsv

Processing 09_PathogenInnateImmuneStress.tsv...
Found headers at row 3. Target column: 'Gene Names'
Saved and sorted: ../output/Validated_09_PathogenInnateImmuneStress.tsv

Processing 06_MechanicalExtracellularStress.tsv...
Found headers at row 3. Target column: 'Gene Names'
Saved and sorted: ../output/Validated_06_MechanicalExtracellularStress.tsv

Processing 02_Proteot

In [7]:
output_folder = "../output/" 
master_output_path = "../output/Master_Tier1_Stress_Targets.tsv"

PILLAR_STRESS_TYPES = {
    "01": "Organelle Stress",
    "02": "Proteotoxic Stress",
    "03": "Genotoxic Stress",
    "04": "Nutrient Stress",
    "05": "Proteasomal Stress",
    "06": "Mechanical Stress",
    "07": "Contact Site Stress",
    "08": "Ribotoxic Stress",
    "09": "Immune Stress",
    "10": "Cell Death Regulators" 
}

In [8]:
# Find all the validated files
validated_files = glob.glob(os.path.join(output_folder, "Validated_*.tsv"))
print(f"\nFound {len(validated_files)} validated TSV files to compile into the master list.")

master_rows = []

for file_path in validated_files:
    file_name = os.path.basename(file_path)
    
    # Identify which pillar number this file belongs to
    pillar_id = next((num for num in PILLAR_STRESS_TYPES if num in file_name), None)
    if not pillar_id:
        continue
    
    
    # Read the validated file
    df = pd.read_csv(file_path, sep='\t')
    
    # Keep rows where a valid PMID was successfully found
    df_tier1 = df[df['Supporting_PMID'].notna() & (df['Supporting_PMID'].astype(str).str.strip() != "")].copy()
    
    if df_tier1.empty:
        continue
        
    # Using dynamic name matching in case formatting varies slightly between files
    prot_col = next((c for c in df_tier1.columns if "Protein" in c), None)
    gene_col = next((c for c in df_tier1.columns if "Gene" in c and "Name" in c), None)
    go_col = next((c for c in df_tier1.columns if "Ontology" in c or "GO" in c), None)
    pmid_col = "Supporting_PMID"
    count_col = "Paper_Count"
    
    if not all([prot_col, gene_col, go_col, count_col]):
        print(f"Warning: Could not align standard columns for {file_name}. Skipping.")
        continue
        
    df_core = df_tier1[[prot_col, gene_col, go_col, pmid_col, count_col]].copy()
    
    df_core.columns = ['Protein Names', 'Gene Names', 'Gene Ontology', 'Supporting_PMID', 'Paper_Count']
    
    df_core['Stress_Type'] = stress_type
    
    # Append to master list accumulator
    master_rows.append(df_core)

# Concatenate all tables together into one long matrix
if master_rows:
    master_df = pd.concat(master_rows, ignore_index=True)
    
    final_column_order = ['Protein Names', 'Gene Names', 'Gene Ontology', 'Supporting_PMID', 'Stress_Type', 'Paper_Count']
    master_df = master_df[final_column_order]
    
    # Group by Stress Type first, then sort by highest literature frequency descending
    master_df = master_df.sort_values(by=['Stress_Type', 'Paper_Count'], ascending=[True, False])
    
    # Write the compiled matrix out cleanly
    master_df.to_csv(master_output_path, sep='\t', index=False)
    
    print(f"Saved total of {len(master_df)} validated targets to: {master_output_path}")
    print(master_df.head(5))
else:
    print("\nNo Tier 1 validated rows with PMIDs were found across any files.")


Found 10 validated TSV files to compile into the master list.
Saved total of 2144 validated targets to: ../output/Master_Tier1_Stress_Targets.tsv
                                          Protein Names  \
1162  Serine/threonine-protein kinase mTOR (EC 2.7.1...   
1200  Serine/threonine-protein kinase mTOR (EC 2.7.1...   
1224  Gasdermin-D (Gasdermin domain-containing prote...   
1107  Long-chain-fatty-acid--CoA ligase 4 (EC 6.2.1....   
1149  NAD-dependent protein deacetylase sirtuin-1 (h...   

                             Gene Names  \
1162  MTOR FRAP FRAP1 FRAP2 RAFT1 RAPT1   
1200  MTOR FRAP FRAP1 FRAP2 RAFT1 RAPT1   
1224        GSDMD DFNA5L GSDMDC1 FKSG10   
1107             ACSL4 ACS4 FACL4 LACS4   
1149                       SIRT1 SIR2L1   

                                          Gene Ontology  Supporting_PMID  \
1162  de novo' pyrimidine nucleobase biosynthetic pr...       42241169.0   
1200  de novo' pyrimidine nucleobase biosynthetic pr...       42241169.0   
1224  ceram